# Notebook 3 -- Modeling

**Purpose**: Build and evaluate machine learning models to predict which side of a retardant drop is protected (pt=1) vs. fire origin (pt=0)

**Input**:

- `training_features_dob.csv` -- DOB pixel statistics (from Notebook 2)
- `training_features_split.csv` -- raster split statistics (from Notebook 2)
- `training_features_expanded.csv` -- expanded statistics (from Notebook 2)
- `training_features_dist` -- distance to unburned pixels statistics (from Notebook 2)


**Approaches**:

1. Simple rule baseline
2. Random Forest - raw DOB features only
3. Random Forest - paired-difference DOB features
4. Random Forest - raw + paired-difference combined 
5. Random Forest - split features only
6. Random Forest - DOB + raster split
7. Random Forest - DOB + split + distance + expanded polygon
8. XGBoost - Full Feature Set





In [1]:
# Importing Necessary Modules
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
RAW, INTERIM, FEATS, MODELS, OUT = (ROOT / p for p in
    ["data/raw", "data/interim", "data/features", "models", "outputs"])
INTERIM.mkdir(parents=True, exist_ok=True)

In [3]:
# Loading extracted features and merging them
# Reloading from saved CSV files to prevent stale varible issues
df_dob  = pd.read_csv(FEATS / "train" / "training_features_dob.csv")
df_split = pd.read_csv(FEATS / "train" / "training_features_split.csv")
df_expanded = pd.read_csv(FEATS / "train" / "training_features_expanded.csv")
df_dist = pd.read_csv(FEATS / "train" / "training_features_dist.csv")

# Mergeing features into DOB features on shared identifiers
df = df_dob.merge(
    df_expanded,
    on=["event_id", "buff_id", "drop_id"],
    how="left"
)
df = df.merge(
    df_split,
    on=["event_id", "buff_id", "drop_id"],
    how="left"
)
df = df.merge(
    df_dist,
    on=["event_id", "buff_id", "drop_id"],
    how="left"
)

print(f"Total rows: {len(df)}")
print(f"Rows with split features: {df['split_burned_pixels'].notna().sum()}")

print(f"PT distribution:\n{df['pt'].value_counts()}")

Total rows: 21081
Rows with split features: 20582
PT distribution:
pt
0    10627
1    10454
Name: count, dtype: int64


In [4]:
# Preparing a Base DataFrame
# Filter to clean drop pairs (exactly 2 rows per drop_id)
# Drops with 1, 3, 4, or 6 rows (~2.4%) are excluded — ambiguous pairs
group_sizes = df.groupby("drop_id").size()
clean_drop_ids = group_sizes[group_sizes == 2].index
df_model = df[df["drop_id"].isin(clean_drop_ids)].copy()
print(f"Clean pairs: {len(df_model)} rows ({df_model['drop_id'].nunique()} drops)")

# Fix NaN mean_dob/median_dob/split_mean_dob — polygons where nothing burned.
# These rows are valid training examples (burned_pixels=0 already captures this).
# Filling with 0 keeps them rather than silently dropping rows.
df_model["mean_dob"]       = df_model["mean_dob"].fillna(0)
df_model["median_dob"]     = df_model["median_dob"].fillna(0)
df_model["split_mean_dob"] = df_model["split_mean_dob"].fillna(0)

# Building Paired-Difference Feature
# For each drop pair, compute how much each side differs from the pair's average.
# Positive diff = more than sibling, negative diff = less than sibling.
# This is the key insight: relative comparison between sides matters more than
# absolute values, because it's self-normalizing across drops of very different sizes.

# DOB paired-difference features (local buffer)
for col in ["unburned_pixels", "early_pixels", "burned_pixels"]:
    df_model[f"{col.replace('_pixels', '')}_pair_diff"] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())


# Split paired-difference features (whole-raster comparison — strongest signal)
for col in ["split_unburned_pixels", "split_early_pixels", "split_burned_pixels"]:
    new_col = col.replace("split_", "split_").replace("_pixels", "") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())


# distance to unburned paired-diff
df_model["dist_pair_diff"] = df_model.groupby("drop_id")["dist_to_unburned"].transform(
    lambda x: x - x.mean()
)

# expanded paired-diff
for col in ["exp_burned_pixels","exp_unburned_pixels","exp_early_pixels"]:
    new_col = col.replace("_pixels","") + "_pair_diff"
    df_model[new_col] = df_model.groupby("drop_id")[col].transform(lambda x: x - x.mean())

print(f"DataFrame shape after feature engineering: {df_model.shape}")

Clean pairs: 20478 rows (10239 drops)
DataFrame shape after feature engineering: (20478, 38)


In [5]:
# Defining Feature Sets

raw_features = [
    "mean_dob", "median_dob", "early_pixels", "late_pixels",
    "unburned_pixels", "burned_pixels", "total_pixels"
]

diff_features = [
    "unburned_pair_diff", "early_pair_diff", "burned_pair_diff"
]

dob_features = raw_features + diff_features


# Whole-raster split features — the strongest predictor discovered.
# Same statistics as DOB features, but computed across the ENTIRE fire raster
# split by the extended drop line, rather than just the small buffer polygon.
split_features = [
    "split_mean_dob", "split_early_pixels", "split_late_pixels",
    "split_unburned_pixels", "split_burned_pixels", "split_total_pixels",
    "split_burned_pair_diff", "split_unburned_pair_diff", "split_early_pair_diff"
]

dist_features = ["dist_to_unburned","dist_pair_diff"]

exp_features = [
    "exp_mean_dob","exp_burned_pixels","exp_unburned_pixels",
    "exp_early_pixels","exp_late_pixels","exp_total_pixels",
    "exp_burned_pair_diff","exp_unburned_pair_diff","exp_early_pair_diff"
]


all_features = dob_features + dist_features + split_features + exp_features


print(all_features)
print(len(all_features))


['mean_dob', 'median_dob', 'early_pixels', 'late_pixels', 'unburned_pixels', 'burned_pixels', 'total_pixels', 'unburned_pair_diff', 'early_pair_diff', 'burned_pair_diff', 'dist_to_unburned', 'dist_pair_diff', 'split_mean_dob', 'split_early_pixels', 'split_late_pixels', 'split_unburned_pixels', 'split_burned_pixels', 'split_total_pixels', 'split_burned_pair_diff', 'split_unburned_pair_diff', 'split_early_pair_diff', 'exp_mean_dob', 'exp_burned_pixels', 'exp_unburned_pixels', 'exp_early_pixels', 'exp_late_pixels', 'exp_total_pixels', 'exp_burned_pair_diff', 'exp_unburned_pair_diff', 'exp_early_pair_diff']
30


In [6]:
# 30 features are used for the final model. 

In [6]:
# Final Clean DataFrame 
# Drop rows with NaN in any feature we'll use.

df_model_clean = df_model.dropna(subset=all_features + ["pt"])
print(f"Rows for training: {len(df_model_clean)}")
print(f"Unique fires: {df_model_clean['event_id'].nunique()}")

# Defining y, groups, gkf ONCE 
# These are defined here and never redefined below.
# All model comparisons use the same split for fair evaluation.
y      = df_model_clean["pt"]      # target: 1=protected, 0=origin
groups = df_model_clean["event_id"] # split by FIRE to prevent leakage
gkf    = GroupKFold(n_splits=5)

Rows for training: 20022
Unique fires: 420


In [7]:
# Model Training and Evaluation Function 
# Trains Random Forest across all 5 GroupKFold folds.
# Accuracy reported as mean ± std across folds for a stable,
# reliable estimate not dependent on one particular split.

def run_model_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []        # accuracy per fold
    importances = [] # feature importances per fold

    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        # 200 trees for stable predictions, random_state=42 for reproducibility
        model = RandomForestClassifier(n_estimators=200, random_state=42)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])     # train on training rows only
        preds = model.predict(X.iloc[test_idx])             # predict on unseen test rows
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        importances.append(model.feature_importances_)

    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    # average feature importances across all 5 folds (axis=0 = average across folds)
    mean_imp = np.mean(importances, axis=0)

    print(f"\n{label}")
    print(f"Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Per-fold: {[round(a, 3) for a in accs]}")
    importance_dict = dict(zip(feature_list, mean_imp))
    print("Top features:", dict(sorted(importance_dict.items(), key=lambda x: x[1], reverse=True)[:5]))
    print("-" * 50)
    return mean_acc, std_acc, model

# Approach 1 - Simple Rule Baseline

Before any ML, we will test the simplest possible rule: **whichever side has more unburned pixels is the protected side.**

Physically motivated - pilots drop retardant ahead of the fire, so the protected side should have more unburned area. Evaluated only on clean drop pairs (exactly 2 sides per drop). 


In [8]:
# Within each drop pair, rank the two sides by unburned_pixels (descending).
# Ranking happens WITHIN each drop_id group — each drop's sides are only
# compared against each other, not against unrelated drops from other fires.
df_rule = df_model_clean.copy()
df_rule["rank_unburned"] = df_rule.groupby("drop_id")["unburned_pixels"].rank(ascending=False)

# rank=1 (more unburned) → predicted protected (1)
# rank=2 (less unburned) → predicted origin (0)
df_rule["rule_pred"] = (df_rule["rank_unburned"] == 1).astype(int)

rule_acc = accuracy_score(df_rule["pt"], df_rule["rule_pred"])
print(f"Rule baseline accuracy: {rule_acc:.3f}")
print(f"Evaluated on {len(df_rule)} rows ({df_rule['drop_id'].nunique()} clean drop pairs)")
# Result: ~68.9% — this is the baseline the ML model must beat

Rule baseline accuracy: 0.689
Evaluated on 20022 rows (10236 clean drop pairs)


# Approaches 2-4 - Random Forest Model Comparisons

Testing four feature configurations to understand what drives accuracy: 

**- Solo unburned_pixels** - can RF match the rule using the same single feature? 

**- Raw features only** - absolute pixel statistics without pairing context

**-Paired-difference only** - relative comparison between sides only

**-Raw + paired-difference** - best of both (our best DOB-only model)


In [9]:
# Test 1: Solo unburned_pixels
# RF using one raw feature can't match the rule's 68.9% because it has no
# awareness of which rows are paired — it tries to find a global threshold.
run_model_cv(["unburned_pixels"], "Solo unburned_pixels (sanity check)")

# Test 2: Raw features only (original baseline)
# RF with 7 features but no pairing context.
# Result: ~68.4% — below the simple rule, confirming raw features alone are insufficient.
run_model_cv(raw_features, "Raw features only")

# Test 3: Paired-difference features only
# Directly encodes 'how does this side compare to its sibling'.
# Result: ~80.0% — large jump confirms relative comparison is the key signal.
run_model_cv(diff_features, "Paired-difference features only")

# Test 4: Raw + paired-difference combined (BEST DOB MODEL)
# Combines absolute pixel statistics with relative paired comparisons.
# Result: ~81.1% ± 0.7% — best DOB-only result.
acc_dob, std_dob, model_dob = run_model_cv(dob_features, "Raw + paired-difference combined (DOB best)")


Solo unburned_pixels (sanity check)
Accuracy: 0.581 ± 0.009
Per-fold: [0.573, 0.575, 0.575, 0.599, 0.581]
Top features: {'unburned_pixels': np.float64(1.0)}
--------------------------------------------------

Raw features only
Accuracy: 0.684 ± 0.012
Per-fold: [0.677, 0.705, 0.672, 0.687, 0.68]
Top features: {'burned_pixels': np.float64(0.2624092970603026), 'unburned_pixels': np.float64(0.1786506469724371), 'mean_dob': np.float64(0.16304163571460895), 'median_dob': np.float64(0.13585405886242857), 'total_pixels': np.float64(0.13245716972391705)}
--------------------------------------------------

Paired-difference features only
Accuracy: 0.800 ± 0.014
Per-fold: [0.779, 0.817, 0.792, 0.812, 0.8]
Top features: {'burned_pair_diff': np.float64(0.5464919252551864), 'unburned_pair_diff': np.float64(0.30161583623390475), 'early_pair_diff': np.float64(0.15189223851090886)}
--------------------------------------------------

Raw + paired-difference combined (DOB best)
Accuracy: 0.811 ± 0.007
P

# Approach 5-7 -- Whole-Raster Split and Expanded Polygon: RF

**Motivation**: DOB and fuel features sampled only the small buffered polygon, which plateaued around 81-82% regardless of feature combination or algorithm choice. The raster split approach instead extends the drop line across the *entire* fire raster, splitting it into two halves and comparing whole-fire burn statistics -- thousands of pixels per side instead of few hundred. Additionally, I also experimented with distance to nearest unburned pixel from the polygons and expanded the polygons to have a middle ground between small polygons and entire raster split. 

**Validated visually in QGIS** on multiple random drops before implementation. 

See Notebook 2 for the extraction function(`extract_split_features`). 

In [10]:
# Test split features alone — whole-fire comparison, no local buffer info
acc_split, std_split, model_split = run_model_cv(split_features, "Split features only")

# Test DOB + split combined — local buffer + whole-fire context together
acc_dob_split, std_dob_split, model_dob_split = run_model_cv(
    dob_features + split_features, "DOB + split combined"
)

# Test everything combined — DOB + distance + split + expanded polygon
acc_all, std_all, model_all = run_model_cv(all_features, "DOB + distance + split + expanded(all features)")


Split features only
Accuracy: 0.840 ± 0.025
Per-fold: [0.804, 0.867, 0.87, 0.826, 0.833]
Top features: {'split_burned_pair_diff': np.float64(0.27670284718628524), 'split_early_pair_diff': np.float64(0.19143803625064826), 'split_unburned_pair_diff': np.float64(0.1341928130685499), 'split_burned_pixels': np.float64(0.10269750714402126), 'split_early_pixels': np.float64(0.0654083528628559)}
--------------------------------------------------

DOB + split combined
Accuracy: 0.898 ± 0.009
Per-fold: [0.886, 0.889, 0.909, 0.908, 0.897]
Top features: {'burned_pair_diff': np.float64(0.20248802249863931), 'split_burned_pair_diff': np.float64(0.1647627253833921), 'split_early_pair_diff': np.float64(0.13462681816921482), 'split_unburned_pair_diff': np.float64(0.08710648168149114), 'unburned_pair_diff': np.float64(0.06104284422238032)}
--------------------------------------------------

DOB + distance + split + expanded(all features)
Accuracy: 0.914 ± 0.010
Per-fold: [0.904, 0.901, 0.923, 0.926, 0.

# Approach 8 -- XGBoost Confirmation

Testing whether a different algorithm can push past the Random Forest result.
If XGBoost matches Random Forest closely, that's strong evidence the ~90% ceiling
is a **data limitation**, not a modeling limitation — i.e. more powerful algorithms
won't help further without genuinely new information (e.g. wind data).

In [11]:
import xgboost as xgb

def run_xgb_cv(feature_list, label):
    X = df_model_clean[feature_list]
    accs = []
    importances = []

    for fold, (train_idx, test_idx) in enumerate(gkf.split(df_model_clean, y, groups)):
        model = xgb.XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric="logloss",
            verbosity=0
        )
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = model.predict(X.iloc[test_idx])
        accs.append(accuracy_score(y.iloc[test_idx], preds))
        importances.append(model.feature_importances_)

    mean_acc = np.mean(accs)
    std_acc  = np.std(accs)
    mean_imp = np.mean(importances, axis=0)

    print(f"\n{label}")
    print(f"Accuracy: {mean_acc:.3f} ± {std_acc:.3f}")
    print(f"Per-fold: {[round(a, 3) for a in accs]}")
    print("Top features:", dict(sorted(zip(feature_list, mean_imp), key=lambda x: x[1], reverse=True)[:5]))
    print("-" * 50)
    return mean_acc, std_acc

acc_xgb, std_xgb = run_xgb_cv(all_features, "XGBoost — All Features")

# Result: XGBoost ≈ Random Forest (~91.5%) : confirms around 90% is a data ceiling,
# not a model choice limitation. XGBoost is the final model (saved below).



XGBoost — All Features
Accuracy: 0.915 ± 0.009
Per-fold: [0.903, 0.905, 0.925, 0.924, 0.918]
Top features: {'exp_burned_pair_diff': np.float32(0.42728144), 'burned_pair_diff': np.float32(0.09380311), 'exp_early_pair_diff': np.float32(0.056008477), 'dist_pair_diff': np.float32(0.043803394), 'split_early_pair_diff': np.float32(0.028666332)}
--------------------------------------------------


---
## Save Random Forest Model

Random Forest trained on DOB + split + distance + expanded features (saved for comparison; the final model is the XGBoost model saved below)

In [12]:
import joblib

final_features = all_features

# Train final model on ALL available training data (not just one fold)
model_final = RandomForestClassifier(n_estimators=200, random_state=42)
model_final.fit(df_model_clean[final_features], y)

joblib.dump(model_final, MODELS / "model_final_rf.pkl")
joblib.dump(final_features, MODELS / "model_features_rf.pkl")
print(f"Model saved. Features used ({len(final_features)}): {final_features}")

Model saved. Features used (30): ['mean_dob', 'median_dob', 'early_pixels', 'late_pixels', 'unburned_pixels', 'burned_pixels', 'total_pixels', 'unburned_pair_diff', 'early_pair_diff', 'burned_pair_diff', 'dist_to_unburned', 'dist_pair_diff', 'split_mean_dob', 'split_early_pixels', 'split_late_pixels', 'split_unburned_pixels', 'split_burned_pixels', 'split_total_pixels', 'split_burned_pair_diff', 'split_unburned_pair_diff', 'split_early_pair_diff', 'exp_mean_dob', 'exp_burned_pixels', 'exp_unburned_pixels', 'exp_early_pixels', 'exp_late_pixels', 'exp_total_pixels', 'exp_burned_pair_diff', 'exp_unburned_pair_diff', 'exp_early_pair_diff']


## Final Model -- 91.5%

Random Forest and XGBoost converge to the same accuracy on the full feature set (DOB + split + distance + expanded polygon). Saving XGBoost as a v2 model

In [14]:
import joblib
final_features = all_features
model_v2 = xgb.XGBClassifier(n_estimators=200, max_depth=6,
                               learning_rate=0.1, random_state=42,
                               eval_metric="logloss", verbosity=0)
model_v2.fit(df_model_clean[final_features], y)
joblib.dump(model_v2, MODELS / "model_final_xgboost.pkl")
joblib.dump(final_features, MODELS / "model_features_xgboost.pkl")
print("Saved!")

Saved!


---
## Updated Results Summary

| Approach | Accuracy |
|---|---|
| Simple rule (unburned_pixels, within-pair) | 68.9% |
| RF -- raw features only | ~68.4% ± 1.2% |
| RF -- paired-difference only | ~80.0% ± 1.4% |
| RF -- raw + paired-difference (DOB best) | 81.1% ± 0.7% |
| RF -- split features only | ~84.0% ± 2.5% | 
| RF -- DOB + raster split | ~89.8% ± 0.9% |
| RF -- DOB + split + distance + expanded polygon | 91.4% ± 1.0% |
| XGBoost -- full feature set (FINAL) | **91.5% ± 0.9%** |

**Where this stands relative to the 95-96% target:** every available data source has now been tried. Random Forest and XGBoost converge on the same
~91-92% ceiling, which is strong evidence this is a **data limitation, not a model-choice limitation**.

**Two remaining hypotheses to close the gap, both requiring the professor's input:**
1. Some fraction of the remaining errors may reflect genuine **mislabeling** in the manually-assigned
   training data (student-labeled 2020-2021 validation data reproduces training accuracy almost exactly,
   which somewhat argues against widespread mislabeling -- but a spot-check on the hardest error cases
   is still worth doing).
   
2. **Wind data** at the time of each drop is the most physically-motivated missing signal (fire spread
   direction is heavily wind-driven). It is implemented in Notebook 2 (Part 4) but not yet used in the model.

DEM terrain features remain a candidate for follow-up if the USGS download issues can be resolved.